# MPU6050 Gesture Model Training

This Google Colab notebook trains **Decision Tree** and **KNN** classifiers for the four gestures `UP`, `DOWN`, `LEFT`, and `RIGHT`. It uses the same 20-sample, 50 ms window and the same 15 features as the ESP32 firmware. The last cell exports a `gesture_model.h` file for on-device inference.

> Collect your own dataset with `firmware/data_collector/data_collector.ino`. Do not report the example notebook's metrics as project results; report only metrics produced from your captured data.

In [ ]:
import io
import numpy as np
import pandas as pd
from google.colab import files
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

uploaded = files.upload()
dataset_name = next(iter(uploaded))
raw = pd.read_csv(io.BytesIO(uploaded[dataset_name]), comment='#')
raw.head()

In [ ]:
required_columns = {'label', 'capture_id', 'sample_index', 'ax', 'ay', 'az'}
missing = required_columns - set(raw.columns)
if missing:
    raise ValueError(f'Missing columns: {sorted(missing)}')

LABEL_TO_ID = {'UP': 0, 'DOWN': 1, 'LEFT': 2, 'RIGHT': 3}
raw['label'] = raw['label'].astype(str).str.strip().str.upper()
unknown = sorted(set(raw['label']) - set(LABEL_TO_ID))
if unknown:
    raise ValueError(f'Unknown labels: {unknown}')

# Reject partial captures so training matches the 20-sample firmware window.
capture_sizes = raw.groupby(['label', 'capture_id']).size()
valid_keys = capture_sizes[capture_sizes == 20].index
valid_key_set = set(valid_keys.tolist())
raw = raw[raw.apply(lambda row: (row['label'], row['capture_id']) in valid_key_set, axis=1)].copy()
raw = raw.sort_values(['label', 'capture_id', 'sample_index'])

FEATURE_NAMES = [
    'ax_mean', 'ay_mean', 'az_mean',
    'ax_std', 'ay_std', 'az_std',
    'ax_min', 'ay_min', 'az_min',
    'ax_max', 'ay_max', 'az_max',
    'ax_delta', 'ay_delta', 'az_delta',
]

def extract_features(group):
    axes = {axis: group[axis].to_numpy(dtype=np.float32) for axis in ('ax', 'ay', 'az')}
    values = []
    values.extend(np.mean(axes[a]) for a in ('ax', 'ay', 'az'))
    values.extend(np.std(axes[a], ddof=0) for a in ('ax', 'ay', 'az'))
    values.extend(np.min(axes[a]) for a in ('ax', 'ay', 'az'))
    values.extend(np.max(axes[a]) for a in ('ax', 'ay', 'az'))
    values.extend(axes[a][-1] - axes[a][0] for a in ('ax', 'ay', 'az'))
    return pd.Series(values, index=FEATURE_NAMES)

feature_rows = []
for (label, capture_id), group in raw.groupby(['label', 'capture_id'], sort=False):
    row = extract_features(group)
    row['label'] = label
    row['capture_id'] = capture_id
    feature_rows.append(row)
features = pd.DataFrame(feature_rows)
counts = features['label'].value_counts().reindex(LABEL_TO_ID, fill_value=0)
display(counts.rename('captures'))
if (counts < 5).any():
    raise ValueError('Collect at least 5 complete captures per gesture; 20-30 is recommended.')

X = features[FEATURE_NAMES].to_numpy(dtype=np.float32)
y = features['label'].map(LABEL_TO_ID).to_numpy(dtype=np.int8)
print(f'Usable gesture captures: {len(X)}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

decision_tree = DecisionTreeClassifier(
    max_depth=6, min_samples_leaf=2, random_state=42
)
decision_tree.fit(X_train, y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
knn_k = min(3, len(X_train_scaled))
knn = KNeighborsClassifier(n_neighbors=knn_k)
knn.fit(X_train_scaled, y_train)

tree_prediction = decision_tree.predict(X_test)
knn_prediction = knn.predict(scaler.transform(X_test))
print(f'Decision Tree test accuracy: {accuracy_score(y_test, tree_prediction):.4f}')
print(f'KNN test accuracy:           {accuracy_score(y_test, knn_prediction):.4f}')
print('\nDecision Tree classification report:')
print(classification_report(y_test, tree_prediction, target_names=list(LABEL_TO_ID), zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(y_test, tree_prediction, display_labels=list(LABEL_TO_ID), ax=axes[0], colorbar=False)
axes[0].set_title('Decision Tree')
ConfusionMatrixDisplay.from_predictions(y_test, knn_prediction, display_labels=list(LABEL_TO_ID), ax=axes[1], colorbar=False)
axes[1].set_title('KNN')
plt.tight_layout()

In [ ]:
# Cross-validation is calculated by capture, not by individual sensor row.
minimum_class_count = int(np.bincount(y).min())
fold_count = min(5, minimum_class_count)
cv = StratifiedKFold(n_splits=fold_count, shuffle=True, random_state=42)
tree_cv = cross_val_score(DecisionTreeClassifier(max_depth=6, min_samples_leaf=2, random_state=42), X, y, cv=cv)
knn_cv = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)), X, y, cv=cv)
print(f'Decision Tree {fold_count}-fold CV: {tree_cv.mean():.4f} +/- {tree_cv.std():.4f}')
print(f'KNN           {fold_count}-fold CV: {knn_cv.mean():.4f} +/- {knn_cv.std():.4f}')

In [ ]:
def c_float(value):
    text = f'{float(value):.9g}'
    if '.' not in text and 'e' not in text.lower():
        text += '.0'
    return text + 'f'

def c_array(values, formatter=str):
    return ', '.join(formatter(value) for value in values)

tree = decision_tree.tree_
leaf_class = []
for node in range(tree.node_count):
    if tree.children_left[node] == tree.children_right[node]:
        leaf_class.append(int(decision_tree.classes_[np.argmax(tree.value[node][0])]))
    else:
        leaf_class.append(-1)

lines = [
    '#pragma once',
    '#include <stdint.h>',
    '#define GESTURE_MODEL_GENERATED 1',
    f'#define GESTURE_FEATURE_COUNT {len(FEATURE_NAMES)}',
    f'static const int16_t DT_LEFT[] = {{{c_array(tree.children_left)}}};',
    f'static const int16_t DT_RIGHT[] = {{{c_array(tree.children_right)}}};',
    f'static const int8_t DT_FEATURE[] = {{{c_array(tree.feature)}}};',
    f'static const float DT_THRESHOLD[] = {{{c_array(tree.threshold, c_float)}}};',
    f'static const int8_t DT_CLASS[] = {{{c_array(leaf_class)}}};',
    'inline int predictDecisionTree(const float *x) {',
    '  int16_t node = 0;',
    '  while (DT_FEATURE[node] >= 0) {',
    '    node = x[DT_FEATURE[node]] <= DT_THRESHOLD[node] ? DT_LEFT[node] : DT_RIGHT[node];',
    '  }',
    '  return DT_CLASS[node];',
    '}',
    f'static const uint16_t KNN_ROWS = {len(X_train_scaled)};',
    f'static const uint8_t KNN_K = {knn_k};',
    f'static const float KNN_MEAN[] = {{{c_array(scaler.mean_, c_float)}}};',
    f'static const float KNN_SCALE[] = {{{c_array(scaler.scale_, c_float)}}};',
    'static const float KNN_X[][GESTURE_FEATURE_COUNT] = {',
]
for row in X_train_scaled:
    lines.append('  {' + c_array(row, c_float) + '},')
lines.extend([
    '};',
    f'static const int8_t KNN_Y[] = {{{c_array(y_train)}}};',
    'inline int predictKnn(const float *x) {',
    '  float bestDistance[KNN_K];',
    '  int8_t bestClass[KNN_K];',
    '  for (uint8_t i = 0; i < KNN_K; ++i) { bestDistance[i] = 1.0e30f; bestClass[i] = -1; }',
    '  for (uint16_t row = 0; row < KNN_ROWS; ++row) {',
    '    float distance = 0.0f;',
    '    for (uint8_t column = 0; column < GESTURE_FEATURE_COUNT; ++column) {',
    '      const float normalized = (x[column] - KNN_MEAN[column]) / KNN_SCALE[column];',
    '      const float difference = normalized - KNN_X[row][column];',
    '      distance += difference * difference;',
    '    }',
    '    for (uint8_t position = 0; position < KNN_K; ++position) {',
    '      if (distance < bestDistance[position]) {',
    '        for (int8_t shift = KNN_K - 1; shift > position; --shift) {',
    '          bestDistance[shift] = bestDistance[shift - 1];',
    '          bestClass[shift] = bestClass[shift - 1];',
    '        }',
    '        bestDistance[position] = distance;',
    '        bestClass[position] = KNN_Y[row];',
    '        break;',
    '      }',
    '    }',
    '  }',
    '  uint8_t votes[4] = {0, 0, 0, 0};',
    '  for (uint8_t i = 0; i < KNN_K; ++i) if (bestClass[i] >= 0 && bestClass[i] < 4) ++votes[bestClass[i]];',
    '  uint8_t winner = 0;',
    '  for (uint8_t label = 1; label < 4; ++label) if (votes[label] > votes[winner]) winner = label;',
    '  return winner;',
    '}',
])

header = '\n'.join(lines) + '\n'
with open('gesture_model.h', 'w', encoding='utf-8') as output:
    output.write(header)
print(f'Exported gesture_model.h ({len(header.encode("utf-8"))} bytes)')
files.download('gesture_model.h')

## Deploy on ESP32

1. Replace `firmware/gesture_remote/gesture_model.h` with the downloaded file.
2. Open `gesture_remote.ino` in Arduino IDE.
3. Keep `USE_KNN_MODEL` set to `0` for Decision Tree or change it to `1` for KNN.
4. Upload to the ESP32 and verify the labels on the OLED.
5. If directions are reversed, capture a new dataset using the final physical orientation of the MPU6050 instead of changing model outputs manually.